# TP3 - Grupo 5

In [ ]:
# importamos paquetes
import pandas as pd
import os
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt  
from scipy.stats import ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as LDA
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, roc_auc_score, RocCurveDisplay, mean_squared_error, precision_score, recall_score, f1_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
# ---------------------------------------------------------------
# Rutas del proyecto. El notebook vive en programs/ y lee de
# ../input/ y escribe en ../output/. Funciona tanto si se ejecuta
# desde programs/ como desde la raiz del TP.
# ---------------------------------------------------------------
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / "input").is_dir() else Path.cwd().parent
INPUT = BASE / "input"
OUTPUT = BASE / "output"
OUTPUT.mkdir(exist_ok=True)

print("Input :", INPUT)
print("Output:", OUTPUT)

In [ ]:
# importamos base de datos (se encuentra en la carpeta de github)
df = pd.read_csv(INPUT / "base.csv", low_memory = False)

### Ejercicio 1: variables para predecir condición de ocupación

In [ ]:
# creamos variable cantidad de personas que viven en el hogar y graficamos
df['PERSONAS_HOGAR'] = df.groupby('CODUSU')['CODUSU'].transform('count')

sns.histplot(df['PERSONAS_HOGAR'].dropna(), kde=False, bins=10)
plt.title('Distribución de la Cantidad de Personas en el Hogar')
plt.xlabel('Cantidad de Personas en el Hogar')
plt.ylabel('Cantidad')
plt.savefig(OUTPUT / "personas-hogar.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# creamos edad al cuadrado y graficamos
df['EDAD2'] = df['CH06']**2

sns.kdeplot(df['EDAD2'].dropna(), fill=True)
plt.title('Densidad de Edad al Cuadrado')
plt.xlabel('Años')
plt.ylabel('Densidad')
plt.savefig(OUTPUT / "edad2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# creamos variable binaria que indica si la persona tiene algún tipo de obra social (no depende de servicios públicos) y graficamos
df['OBRA_SOCIAL'] = ((df['CH08'] == 1) | (df['CH08'] == 2) | (df['CH08'] == 12)).astype(int)

sns.countplot(x=df['OBRA_SOCIAL'].replace({1: 'Sí', 0: 'No'}))
plt.title('Distribución de Dependencia a Servicios Públicos')
plt.xlabel('Depende de Servicios de Salud del Estado')
plt.ylabel('Cantidad')
plt.savefig(OUTPUT / "obra-social.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# volvemos a definir variable para calcular años de educación
primaria = 6
egb = 9
secundaria = 12
terciario = 14
universidad = 16
posgrado = 18

def calcular_educacion(fila):
    nivel = fila['CH12']
    finalizado = fila['CH13']
    aprobado = fila['CH14']
    educacion = fila['NIVEL_ED']

    if nivel == 1:
        return 0
    elif nivel == 2:
        if finalizado == 1:
            return primaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 2:
                    return primaria
                else: return primaria / 2 # estandarizámos
            return aprobado
    elif nivel == 3:
        if finalizado == 1:
            return egb
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 2:
                    return egb
                else: return egb / 2 # estandarizámos
            return aprobado
    elif nivel == 4:
        if finalizado == 1:
            return secundaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 4:
                    return secundaria
                else: return primaria # lo último que sí sabemos que aprobó
            return primaria + aprobado
    elif nivel == 5:
        if finalizado == 1:
            return secundaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 4:
                    return secundaria
                else: return egb # lo último que sí sabemos que aprobó
            return egb + aprobado
    elif nivel == 6:
        if finalizado == 1:
            return terciario
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 6:
                    return terciario
                else: return secundaria # lo último que sí sabemos que aprobó
            return secundaria + aprobado
    elif nivel == 7:
        if finalizado == 1:
            return universidad
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                if educacion == 6:
                    return universidad
                else: return secundaria # lo último que sí sabemos que aprobó
            return secundaria + aprobado
    elif nivel == 8:
        if finalizado == 1:
            return posgrado
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return universidad # lo último que sí sabemos que aprobó
            return universidad + aprobado
    else:
        return 0

In [ ]:
# creamos cantidad de años de educación y graficamos
df['EDUCACION'] = df.apply(calcular_educacion, axis=1)

sns.histplot(df['EDUCACION'].dropna(), kde=True, bins=15)
plt.title('Distribución de Cantidad de Años de Educación')
plt.xlabel('Años')
plt.ylabel('Densidad')
plt.savefig(OUTPUT / "educacion.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# creamos interacción edad y años de educación, graficamos
df['EDADxEDUCACION'] = df['CH06'] * df['EDUCACION']

plt.figure(figsize=(8, 5))
sns.boxplot(x=df.loc[df['ESTADO'].isin([1, 2, 3]), 'ESTADO'].replace({1: 'Ocupado', 2: 'Desocupado', 3: 'Inactivo'}), 
            y=df.loc[df['ESTADO'].isin([1, 2, 3]), 'EDADxEDUCACION'])
plt.title('Interacción Edad y Educación por Condición de Ocupación')
plt.xlabel('')
plt.ylabel('Edad x Años de Educación')
plt.savefig(OUTPUT / "edadxeducacion.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# creamos variable binaria que indica si asistió a colegio privado (proxy nivel económico padres) y graficamos
df['COLEGIO_PRIVADO'] = (df['CH11'] == 2).astype(int)

sns.countplot(x='COLEGIO_PRIVADO', data=df)
plt.title('Distribución de Asistencia a Colegio Privado')
plt.xlabel('Colegio privado (1=Sí, 0=No)')
plt.ylabel('Cantidad')
plt.savefig(OUTPUT / "colegio-privado.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# creamos variable constante para usar de columna de unos
df['CONSTANTE'] = 1

In [ ]:
# creamos bases 'respondieron' y 'norespondieron', agregamos variable DESOCUPADOS
respondieron = pd.DataFrame(df[df['ESTADO'] != 0])
norespondieron = pd.DataFrame(df[df['ESTADO'] == 0])
respondieron['DESOCUPADOS'] = (respondieron['ESTADO'] == 2).astype(int)

# exportamos: es el insumo del TP4 y del TP5
respondieron.to_csv(OUTPUT / "respondieron.csv", index=False)
print("respondieron.csv guardada:", respondieron.shape)

In [ ]:
# la lista de variables relevantes para el análisis entonces es:
variables = ['CONSTANTE', 'CH04', 'CH06', 'CH07', 'CH16', 'PERSONAS_HOGAR', 'EDAD2', 'OBRA_SOCIAL', 'EDUCACION', 'EDADxEDUCACION', 'COLEGIO_PRIVADO']

### Ejercicio 2: tabla de diferencia de medias 

In [ ]:
# creamos matriz X y vector y para 2004
X_04 = respondieron.loc[respondieron['ANO4'] == 2004, variables].to_numpy()
y_04 = respondieron.loc[respondieron['ANO4'] == 2004, 'DESOCUPADOS'].to_numpy()

In [ ]:
# creamos matriz X y vector y para 2024
X_24 = respondieron.loc[respondieron['ANO4'] == 2024, variables].to_numpy()
y_24 = respondieron.loc[respondieron['ANO4'] == 2024, 'DESOCUPADOS'].to_numpy()

In [ ]:
# separamos en base entrenamiento y base testeo
X_train_04, X_test_04, y_train_04, y_test_04 = train_test_split(X_04, y_04, test_size=0.3, random_state=444)
X_train_24, X_test_24, y_train_24, y_test_24 = train_test_split(X_24, y_24, test_size=0.3, random_state=444)

In [ ]:
# calculamos medias y diferencia 2004
medias_train_04 = []
medias_test_04 = []
diferencias_04 = []

for i in range(len(variables)):
    train = X_train_04[:, i]
    test = X_test_04[:, i]

    media_train = np.nanmean(train)
    media_test = np.nanmean(test)
    
    medias_train_04.append(media_train)
    medias_test_04.append(media_test)
    diferencias_04.append(media_train - media_test)

In [ ]:
# armamos tabla 2004
tabla_medias = pd.DataFrame({'Variable': variables, 'Media Train': medias_train_04, 
                             'Media Test': medias_test_04, 'Diferencia': diferencias_04})

In [ ]:
# redondeamos y exportamos a Latex
tabla_medias = tabla_medias.round(2)
tabla_latex = tabla_medias.to_latex(index=False, caption="Medias y diferencias muestras de entrenamiento y de testeo", label="tab:medias", float_format="%.2f")

with open(OUTPUT / "tabla-medias-04.tex", "w") as f:
    f.write(tabla_latex)

In [ ]:
# calculamos medias y diferencia 2024
medias_train_24 = []
medias_test_24 = []
diferencias_24 = []

for i in range(len(variables)):
    train = X_train_24[:, i]
    test = X_test_24[:, i]

    media_train = np.nanmean(train)
    media_test = np.nanmean(test)
    
    medias_train_24.append(media_train)
    medias_test_24.append(media_test)
    diferencias_24.append(media_train - media_test)

In [ ]:
# armamos tabla 2024
tabla_medias = pd.DataFrame({'Variable': variables, 'Media Train': medias_train_24, 
                             'Media Test': medias_test_24, 'Diferencia': diferencias_24})

In [ ]:
# redondeamos y exportamos a Latex
tabla_medias = tabla_medias.round(2)
tabla_latex = tabla_medias.to_latex(index=False, caption="Medias y diferencias muestras de entrenamiento y de testeo", label="tab:medias", float_format="%.2f")

with open(OUTPUT / "tabla-medias-24.tex", "w") as f:
    f.write(tabla_latex)

### Ejercicio 3: implementación de metodos

#### Regresión Logística

##### 2004

In [ ]:
# modelamos
log_reg_04 = LDA(penalty=None, max_iter=10000).fit(X_train_04, y_train_04)
y_predRL_04 = log_reg_04.predict(X_test_04)
y_probRL_04 = log_reg_04.predict_proba(X_test_04)[:,1]

matriz_confusionRL_04 = confusion_matrix(y_test_04, y_predRL_04)
accuracyRL_04 = accuracy_score(y_test_04, y_predRL_04)
precisionRL_04 = precision_score(y_test_04, y_predRL_04, zero_division=1)
recallRL_04 = recall_score(y_test_04, y_predRL_04)
f1RL_04 = f1_score(y_test_04, y_predRL_04)
aucRL_04 = roc_auc_score(y_test_04, y_probRL_04)
mseRL_04 = mean_squared_error(y_test_04, y_predRL_04)
fprRL_04, tprRL_04, thresholdsRL_04 = roc_curve(y_test_04, y_probRL_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionRL_04}')
print(f'El accuracy score es: {accuracyRL_04}.')
print(f'La precisión es: {precisionRL_04}.')
print(f'El recall es: {recallRL_04}.')
print(f'El F1-score es: {f1RL_04}.')
print(f'El área bajo la curva ROC es: {aucRL_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseRL_04}')

In [ ]:
# graficamos su curva ROC
displayRL_04 = RocCurveDisplay(fpr=fprRL_04, tpr=tprRL_04, roc_auc=aucRL_04, estimator_name='RL')
displayRL_04.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "reg-log-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelamos
log_reg_24 = LDA(penalty=None, max_iter=2000).fit(X_train_24, y_train_24)
y_predRL_24 = log_reg_24.predict(X_test_24)
y_probRL_24 = log_reg_24.predict_proba(X_test_24)[:,1]

matriz_confusionRL_24 = confusion_matrix(y_test_24, y_predRL_24)
accuracyRL_24 = accuracy_score(y_test_24, y_predRL_24)
precisionRL_24 = precision_score(y_test_24, y_predRL_24)
recallRL_24 = recall_score(y_test_24, y_predRL_24)
f1RL_24 = f1_score(y_test_24, y_predRL_24)
aucRL_24 = roc_auc_score(y_test_24, y_probRL_24)
mseRL_24 = mean_squared_error(y_test_24, y_predRL_24)
fprRL_24, tprRL_24, thresholdsRL_24 = roc_curve(y_test_24, y_probRL_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionRL_24}')
print(f'El accuracy score es: {accuracyRL_24}.')
print(f'La precisión es: {precisionRL_24}.')
print(f'El recall es: {recallRL_24}.')
print(f'El F1-score es: {f1RL_24}.')
print(f'El área bajo la curva ROC es: {aucRL_24}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseRL_24}')

In [ ]:
# graficamos su curva ROC
displayRL_24 = RocCurveDisplay(fpr=fprRL_24, tpr=tprRL_24, roc_auc=aucRL_24, estimator_name='RL')
displayRL_24.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "reg-log-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### LDA - análisis discriminante lineal

##### 2004

In [ ]:
# modelamos
lda_04 = LDA(max_iter=5000).fit(X_train_04, y_train_04)
y_predLDA_04 = lda_04.predict(X_test_04)
y_probLDA_04 = lda_04.predict_proba(X_test_04)[:,1]

matriz_confusionLDA_04 = confusion_matrix(y_test_04, y_predLDA_04)
accuracyLDA_04 = accuracy_score(y_test_04, y_predLDA_04)
precisionLDA_04 = precision_score(y_test_04, y_predLDA_04)
recallLDA_04 = recall_score(y_test_04, y_predLDA_04)
f1LDA_04 = f1_score(y_test_04, y_predLDA_04)
aucLDA_04 = roc_auc_score(y_test_04, y_probLDA_04)
mseLDA_04 = mean_squared_error(y_test_04, y_predLDA_04)
fprLDA_04, tprLDA_04, thresholdsLDA_04 = roc_curve(y_test_04, y_probLDA_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionLDA_04}')
print(f'El accuracy score es: {accuracyLDA_04}.')
print(f'La precisión es: {precisionLDA_04}.')
print(f'El recall es: {recallLDA_04}.')
print(f'El F1-score es: {f1LDA_04}.')
print(f'El área bajo la curva ROC es: {aucLDA_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseLDA_04}')

In [ ]:
# graficamos su curva ROC
displayLDA_04 = RocCurveDisplay(fpr=fprLDA_04, tpr=tprLDA_04, roc_auc=aucLDA_04, estimator_name='LDA')
displayLDA_04.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "LDA-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelamos
lda_24 = LDA(max_iter=5000).fit(X_train_24, y_train_24)
y_predLDA_24 = lda_24.predict(X_test_24)
y_probLDA_24 = lda_24.predict_proba(X_test_24)[:,1]

matriz_confusionLDA_24 = confusion_matrix(y_test_24, y_predLDA_24)
accuracyLDA_24 = accuracy_score(y_test_24, y_predLDA_24)
precisionLDA_24 = precision_score(y_test_24, y_predLDA_24)
recallLDA_24 = recall_score(y_test_24, y_predLDA_24)
f1LDA_24 = f1_score(y_test_24, y_predLDA_24)
aucLDA_24 = roc_auc_score(y_test_24, y_probLDA_24)
mseLDA_24 = mean_squared_error(y_test_24, y_predLDA_24)
fprLDA_24, tprLDA_24, thresholdsLDA_24 = roc_curve(y_test_24, y_probLDA_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionLDA_24}')
print(f'El accuracy score es: {accuracyLDA_24}.')
print(f'La precisión es: {precisionLDA_24}.')
print(f'El recall es: {recallLDA_24}.')
print(f'El F1-score es: {f1LDA_24}.')
print(f'El área bajo la curva ROC es: {aucLDA_24}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseLDA_24}')

In [ ]:
# graficamos su curva ROC
displayLDA_24 = RocCurveDisplay(fpr=fprLDA_24, tpr=tprLDA_24, roc_auc=aucLDA_24, estimator_name='LDA')
displayLDA_24.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "LDA-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### QDA - análisis discriminante cuadratico

##### 2004

In [ ]:
# modelamos
qda_04 = QDA(reg_param=0.01).fit(X_train_04, y_train_04)
y_predQDA_04 = qda_04.predict(X_test_04)
y_probQDA_04 = qda_04.predict_proba(X_test_04)[:,1]

matriz_confusionQDA_04 = confusion_matrix(y_test_04, y_predQDA_04)
accuracyQDA_04 = accuracy_score(y_test_04, y_predQDA_04)
precisionQDA_04 = precision_score(y_test_04, y_predQDA_04)
recallQDA_04 = recall_score(y_test_04, y_predQDA_04)
f1QDA_04 = f1_score(y_test_04, y_predQDA_04)
aucQDA_04 = roc_auc_score(y_test_04, y_probQDA_04)
mseQDA_04 = mean_squared_error(y_test_04, y_predQDA_04)
fprQDA_04, tprQDA_04, thresholdsQDA_04 = roc_curve(y_test_04, y_probQDA_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionQDA_04}')
print(f'El accuracy score es: {accuracyQDA_04}.')
print(f'La precisión es: {precisionQDA_04}.')
print(f'El recall es: {recallQDA_04}.')
print(f'El F1-score es: {f1QDA_04}.')
print(f'El área bajo la curva ROC es: {aucQDA_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseQDA_04}')

In [ ]:
# graficamos su curva ROC
displayQDA_04 = RocCurveDisplay(fpr=fprQDA_04, tpr=tprQDA_04, roc_auc=aucQDA_04, estimator_name='QDA')
displayQDA_04.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "QDA-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelamos
qda_24 = QDA(reg_param=0.01).fit(X_train_24, y_train_24)
y_predQDA_24 = qda_24.predict(X_test_24)
y_probQDA_24 = qda_24.predict_proba(X_test_24)[:,1]

matriz_confusionQDA_24 = confusion_matrix(y_test_24, y_predQDA_24)
accuracyQDA_24 = accuracy_score(y_test_24, y_predQDA_24)
precisionQDA_24 = precision_score(y_test_24, y_predQDA_24)
recallQDA_24 = recall_score(y_test_24, y_predQDA_24)
f1QDA_24 = f1_score(y_test_24, y_predQDA_24)
aucQDA_24 = roc_auc_score(y_test_24, y_probQDA_24)
mseQDA_24 = mean_squared_error(y_test_24, y_predQDA_24)
fprQDA_24, tprQDA_24, thresholdsQDA_24 = roc_curve(y_test_24, y_probQDA_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionQDA_04}')
print(f'El accuracy score es: {accuracyQDA_04}.')
print(f'La precisión es: {precisionQDA_04}.')
print(f'El recall es: {recallQDA_04}.')
print(f'El F1-score es: {f1QDA_04}.')
print(f'El área bajo la curva ROC es: {aucQDA_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseQDA_04}')

In [ ]:
# graficamos su curva ROC
displayQDA_24 = RocCurveDisplay(fpr=fprQDA_24, tpr=tprQDA_24, roc_auc=aucQDA_24, estimator_name='QDA')
displayQDA_24.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "QDA-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### KNN - vecinos cercanos

##### 2004

In [ ]:
# modelamos usando K = 5
knn_04 = KNeighborsClassifier(n_neighbors=5).fit(X_train_04, y_train_04)
y_predKNN_04 = knn_04.predict(X_test_04)
y_probKNN_04 = knn_04.predict_proba(X_test_04)[:,1]

matriz_confusionKNN_04 = confusion_matrix(y_test_04, y_predKNN_04)
accuracyKNN_04 = accuracy_score(y_test_04, y_predKNN_04)
precisionKNN_04 = precision_score(y_test_04, y_predKNN_04)
recallKNN_04 = recall_score(y_test_04, y_predKNN_04)
f1KNN_04 = f1_score(y_test_04, y_predKNN_04)
aucKNN_04 = roc_auc_score(y_test_04, y_probKNN_04)
mseKNN_04 = mean_squared_error(y_test_04, y_predKNN_04)
fprKNN_04, tprKNN_04, thresholdsKNN_04 = roc_curve(y_test_04, y_probKNN_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionKNN_04}')
print(f'El accuracy score es: {accuracyKNN_04}.')
print(f'La precisión es: {precisionKNN_04}.')
print(f'El recall es: {recallKNN_04}.')
print(f'El F1-score es: {f1KNN_04}.')
print(f'El área bajo la curva ROC es: {aucKNN_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseKNN_04}')

In [ ]:
# graficamos su curva ROC
displayKNN_04 = RocCurveDisplay(fpr=fprKNN_04, tpr=tprKNN_04, roc_auc=aucKNN_04, estimator_name='KNN')
displayKNN_04.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "KNN-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelamos usando K = 5
knn_24 = KNeighborsClassifier(n_neighbors=5).fit(X_train_24, y_train_24)
y_predKNN_24 = knn_24.predict(X_test_24)
y_probKNN_24 = knn_24.predict_proba(X_test_24)[:,1]

matriz_confusionKNN_24 = confusion_matrix(y_test_24, y_predKNN_24)
accuracyKNN_24 = accuracy_score(y_test_24, y_predKNN_24)
precisionKNN_24 = precision_score(y_test_24, y_predKNN_24)
recallKNN_24 = recall_score(y_test_24, y_predKNN_24)
f1KNN_24 = f1_score(y_test_24, y_predKNN_24)
aucKNN_24 = roc_auc_score(y_test_24, y_probKNN_24)
mseKNN_24 = mean_squared_error(y_test_24, y_predKNN_24)
fprKNN_24, tprKNN_24, thresholdsKNN_24 = roc_curve(y_test_24, y_probKNN_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionKNN_24}')
print(f'El accuracy score es: {accuracyKNN_24}.')
print(f'La precisión es: {precisionKNN_24}.')
print(f'El recall es: {recallKNN_24}.')
print(f'El F1-score es: {f1KNN_24}.')
print(f'El área bajo la curva ROC es: {aucKNN_24}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseKNN_24}')

In [ ]:
# graficamos su curva ROC
displayKNN_24 = RocCurveDisplay(fpr=fprKNN_24, tpr=tprKNN_24, roc_auc=aucKNN_24, estimator_name='KNN')
displayKNN_24.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "KNN-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### Naive Bayes

##### 2004

In [ ]:
# modelamos
nb_04 = GaussianNB().fit(X_train_04, y_train_04)
y_predNB_04 = nb_04.predict(X_test_04)
y_probNB_04 = nb_04.predict_proba(X_test_04)[:,1]

matriz_confusionNB_04 = confusion_matrix(y_test_04, y_predNB_04)
accuracyNB_04 = accuracy_score(y_test_04, y_predNB_04)
precisionNB_04 = precision_score(y_test_04, y_predNB_04)
recallNB_04 = recall_score(y_test_04, y_predNB_04)
f1NB_04 = f1_score(y_test_04, y_predNB_04)
aucNB_04 = roc_auc_score(y_test_04, y_probNB_04)
mseNB_04 = mean_squared_error(y_test_04, y_predNB_04)
fprNB_04, tprNB_04, thresholdsNB_04 = roc_curve(y_test_04, y_probNB_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionNB_04}')
print(f'El accuracy score es: {accuracyNB_04}.')
print(f'La precisión es: {precisionNB_04}.')
print(f'El recall es: {recallNB_04}.')
print(f'El F1-score es: {f1NB_04}.')
print(f'El área bajo la curva ROC es: {aucNB_04}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseNB_04}')

In [ ]:
# graficamos su curva ROC
displayNB_04 = RocCurveDisplay(fpr=fprNB_04, tpr=tprNB_04, roc_auc=aucNB_04, estimator_name='NB')
displayNB_04.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "NB-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelamos
nb_24 = GaussianNB().fit(X_train_24, y_train_24)
y_predNB_24 = nb_24.predict(X_test_24)
y_probNB_24 = nb_24.predict_proba(X_test_24)[:,1]

matriz_confusionNB_24 = confusion_matrix(y_test_24, y_predNB_24)
accuracyNB_24 = accuracy_score(y_test_24, y_predNB_24)
precisionNB_24 = precision_score(y_test_24, y_predNB_24)
recallNB_24 = recall_score(y_test_24, y_predNB_24)
f1NB_24 = f1_score(y_test_24, y_predNB_24)
aucNB_24 = roc_auc_score(y_test_24, y_probNB_24)
mseNB_24 = mean_squared_error(y_test_24, y_predNB_24)
fprNB_24, tprNB_24, thresholdsNB_24 = roc_curve(y_test_24, y_probNB_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionNB_24}')
print(f'El accuracy score es: {accuracyNB_24}.')
print(f'La precisión es: {precisionNB_24}.')
print(f'El recall es: {recallNB_24}.')
print(f'El F1-score es: {f1NB_24}.')
print(f'El área bajo la curva ROC es: {aucNB_24}')
print(f'El error cuadrático medio de la muestra de testeo es: {mseNB_24}')

In [ ]:
# graficamos su curva ROC
displayNB_24 = RocCurveDisplay(fpr=fprNB_24, tpr=tprNB_24, roc_auc=aucNB_24, estimator_name='NB')
displayNB_24.plot()
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "NB-2024.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# guardo matrices de confusión a Latex
matrices = ["matriz_confusionRL_04", "matriz_confusionRL_24", "matriz_confusionLDA_04", "matriz_confusionLDA_24", "matriz_confusionQDA_04",
            "matriz_confusionQDA_24", "matriz_confusionKNN_04", "matriz_confusionKNN_24", "matriz_confusionNB_04", "matriz_confusionNB_24"]

for matriz in matrices:
    data = globals()[matriz]  # Acceder a la variable por su nombre
    cm_df = pd.DataFrame(data, index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"])
    #print(f"% {matriz}")
    #print(cm_df.to_latex())

### Ejercicio 5: predicciones base 'norespondieron'

##### 2024 - no hay ninguna observación en 2004

In [ ]:
# predecimos con los modelos QDA y Regresión Logística
X_nuevo = norespondieron[variables].to_numpy()

y_predQDA = qda_24.predict(X_nuevo)
y_predRL = log_reg_24.predict(X_nuevo)

In [ ]:
# proporción desocupados en QDA
desocupados_QDA = np.count_nonzero(y_predQDA == 1)
print(f'El modelo QDA predice que, de los que no respondieron, {(desocupados_QDA/len(y_predQDA))*100}% serán desocupados.')

In [ ]:
# proporción desocupados en Regresión Logística
desocupados_RL = np.count_nonzero(y_predRL == 1)
print(f'El modelo de regresión logística predice que, de los que no respondieron, {(desocupados_RL/len(y_predRL))*100}% serán desocupados.')

FIN :)